# Operational outputs — O01 and O02

This notebook shows how saved detection and diagnosis results are translated into staff-facing outputs. These commands do not train a model or change headline experiment metrics.

## Create the outputs once

Run these after D07 and G03a finish, replacing the placeholders with the timestamped folders printed by those runs:

```bash
sfsa events --run-dir results/final_runs/<D07-run-folder> --output-name O02_events
sfsa calibrate-diagnosis --run-dir results/final_runs/<G03a-run-folder> --policy configs/operational_policies/O01_reliability_gated.yaml
sfsa prioritise-events --events-dir results/final_runs/<D07-run-folder>/O02_events --diagnosis-dir results/final_runs/<G03a-run-folder>/O01_reliability_gated
```

O01 applies the 70% reliability gate. O02 applies the documented severity, duration, and equipment-criticality policy.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'final_runs'

def latest_run(experiment):
    matches = sorted(RESULTS.glob(f'*_{experiment}'))
    if not matches:
        raise FileNotFoundError(f'No completed run found for {experiment}.')
    return matches[-1]

d07_run = latest_run('D07_xgboost_detection')
g03a_run = latest_run('G03a_xgboost_instrumented_bms')
events_dir = d07_run / 'O02_events'
diagnosis_dir = g03a_run / 'O01_reliability_gated'
priority_dir = events_dir / 'O02_priority_cards'

for path in [events_dir, diagnosis_dir, priority_dir]:
    if not path.exists():
        raise FileNotFoundError(f'Missing operational output: {path}. Run the commands above first.')

print(f'D07 source run: {d07_run.name}')
print(f'G03a source run: {g03a_run.name}')

D07 source run: 20261026T141733Z_rbc_ashrae1312_real_D07_xgboost_detection
G03a source run: 20261026T153634Z_rbc_ashrae1312_real_G03a_xgboost_instrumented_bms


In [2]:
event_summary = json.loads((events_dir / 'event_summary.json').read_text())
diagnosis_summary = json.loads((diagnosis_dir / 'summary.json').read_text())
priority_summary = json.loads((priority_dir / 'priority_summary.json').read_text())

display(pd.DataFrame([event_summary]))
display(pd.DataFrame([diagnosis_summary]))
display(pd.DataFrame([priority_summary]))

,event_count,events_by_outcome,patterns_run,policy,source_run
0,49,{'true_detection': 49},None,"{'gap_tolerance_minutes': 5, 'minimum_alert_ro...",results/final_runs/20261026T141733Z_rbc_ashrae...


,anomaly_reports,fault_group_shown,fault_group_uncertain,fault_reports,fault_type_shown,fault_type_uncertain,limitation,minimum_reliability,minimum_reports,report_unit
0,40,17,23,40,10,30,Reliability is descriptive cross-validated evi...,0.7,5,one held-out date/scenario fault session


,diagnosis_available,event_count,limitation,policy,priority_bands,source_diagnoses,source_events
0,17,49,Severity and criticality are configurable prot...,{'actions': {'critical': 'Escalate immediately...,{'High': 49},results/final_runs/20261026T153634Z_rbc_ashrae...,results/final_runs/20261026T141733Z_rbc_ashrae...


In [3]:
diagnosis_reports = pd.read_csv(diagnosis_dir / 'diagnosis_reports.csv')
priority_cards = pd.read_csv(priority_dir / 'priority_cards.csv')

print('Reliability-gated diagnosis reports')
display(diagnosis_reports.head(10))
print('Example priority cards')
display(priority_cards.head(10))

Reliability-gated diagnosis reports


,fold,date_folder,scenario_file,rows,true_fault_group,true_fault_type,predicted_fault_group,predicted_fault_type,group_evidence_reports,group_correct_reports,group_observed_reliability,group_reliable,type_evidence_reports,type_correct_reports,type_observed_reliability,type_reliable,status,fault_group,fault_type
0,1,20070820,EADamStuck_100.csv,1440,airside_damper,exhaust_air_damper_stuck,airside_damper,exhaust_air_damper_stuck,21.0,11.0,0.52381,False,15.0,5.0,0.333333,False,Anomaly detected,Uncertain,Uncertain
1,1,20070820,baseline.csv,1440,normal,normal,normal,normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Normal,Not applicable,Not applicable
2,2,20070821,EADamStuck_0.csv,1440,airside_damper,exhaust_air_damper_stuck,airside_damper,exhaust_air_damper_stuck,21.0,11.0,0.52381,False,15.0,5.0,0.333333,False,Anomaly detected,Uncertain,Uncertain
3,2,20070821,baseline.csv,1440,normal,normal,normal,normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Normal,Not applicable,Not applicable
4,3,20070822,RetFanFixed_30.csv,1440,return_fan,return_fan_fixed_speed,airside_damper,exhaust_air_damper_stuck,21.0,11.0,0.52381,False,15.0,5.0,0.333333,False,Anomaly detected,Uncertain,Uncertain
5,3,20070822,baseline.csv,1440,normal,normal,normal,normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Normal,Not applicable,Not applicable
6,4,20070823,RetFan_Fail.csv,1440,return_fan,return_fan_failure,airside_damper,exhaust_air_damper_stuck,21.0,11.0,0.52381,False,15.0,5.0,0.333333,False,Anomaly detected,Uncertain,Uncertain
7,4,20070823,baseline.csv,1440,normal,normal,normal,normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Normal,Not applicable,Not applicable
8,5,20070826,OADamStuck_0.csv,1440,airside_damper,outdoor_air_damper_stuck,airside_damper,exhaust_air_damper_stuck,21.0,11.0,0.52381,False,15.0,5.0,0.333333,False,Anomaly detected,Uncertain,Uncertain
9,5,20070826,baseline.csv,1440,normal,normal,normal,normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Normal,Not applicable,Not applicable


Example priority cards


,event_id,fold,model,date_folder,scenario_file,fault_type,fault_intensity,start_timestamp,end_timestamp,duration_minutes,...,severity_label,severity_basis,duration_level,duration_label,asset_key,asset_criticality_level,asset_criticality_label,priority_score,priority_band,recommended_action
0,xgboost-f6-20070827-CooCoiValStuck_0-000,6,xgboost,20070827,CooCoiValStuck_0.csv,cooling_coil_valve_stuck,0,2007-08-27,2007-08-27 23:59:00,1440.0,...,Cooling-capacity impact,fault type,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
1,xgboost-f23-20070828-HeaCoiValLeak_0.4-000,23,xgboost,20070828,HeaCoiValLeak_0.4.csv,Uncertain,k_0.4,2007-08-28,2007-08-28 23:59:00,1440.0,...,Heating-capacity or comfort impact,fault group,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
2,xgboost-f22-20070829-HeaCoiValLea_1-000,22,xgboost,20070829,HeaCoiValLea_1.csv,Uncertain,1,2007-08-29,2007-08-29 23:59:00,1440.0,...,Heating-capacity or comfort impact,fault group,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
3,xgboost-f21-20070830-HeaCoiValLa_2-000,21,xgboost,20070830,HeaCoiValLa_2.csv,cooling_coil_valve_stuck,2,2007-08-30,2007-08-30 23:59:00,1440.0,...,Cooling-capacity impact,fault type,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
4,xgboost-f7-20070831-CooCoiValStuck_100-000,7,xgboost,20070831,CooCoiValStuck_100.csv,cooling_coil_valve_stuck,100,2007-08-31,2007-08-31 23:59:00,1440.0,...,Cooling-capacity impact,fault type,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
5,xgboost-f8-20070901-CooCoiValStuck_15-000,8,xgboost,20070901,CooCoiValStuck_15.csv,cooling_coil_valve_stuck,15,2007-09-01,2007-09-01 23:59:00,1440.0,...,Cooling-capacity impact,fault type,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
6,xgboost-f10-20070902-CooCoiValStuck_65-000,10,xgboost,20070902,CooCoiValStuck_65.csv,cooling_coil_valve_stuck,65,2007-09-02,2007-09-02 23:59:00,1440.0,...,Cooling-capacity impact,fault type,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
7,xgboost-f16-20080203-EADamStuck_0-000,16,xgboost,20080203,EADamStuck_0.csv,Uncertain,0,2008-02-03,2008-02-03 23:59:00,1440.0,...,Heating-capacity or comfort impact,fault group,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
8,xgboost-f20-20080206-HeaCoiFou_2-000,20,xgboost,20080206,HeaCoiFou_2.csv,Uncertain,2,2008-02-06,2008-02-06 23:59:00,1440.0,...,Heating-capacity or comfort impact,fault group,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.
9,xgboost-f25-20080207-HeatCoiReduCapa_1-000,25,xgboost,20080207,HeatCoiReduCapa_1.csv,Uncertain,1,2008-02-07,2008-02-07 23:59:00,1440.0,...,Heating-capacity or comfort impact,fault group,5,Over 8 hours,rbc_ers_ahu,3,Medium (project assumption; client confirmatio...,77.5,High,Investigate today.


## Interpretation limit

The event durations are durations of labelled and flagged sessions. Because the source dataset labels faults for an entire recording day, they are not verified real-world fault durations or onset times. Use these outputs to demonstrate the pipeline and show a small number of example alert cards; do not present priority-band counts as deployment performance.